# Raw Data to SQLite Databases

This notebook rebuilds or updates the SQLite databases in `Outputs/` from the raw files in `Raw/`.


In [1]:
# Title: Setup paths and imports
# Documentation: Load the packages and define the folders used by all later cells.
from pathlib import Path
import re
import sqlite3
import unicodedata

import pandas as pd

root = Path.cwd()
raw = root / "Raw"
out = root / "Outputs"
out.mkdir(exist_ok=True)

baci_raw = raw / "BACI_HS92_V202601"
wgi_file = raw / "wgidataset_with_sourcedata-2025.xlsx"
wmd_file = raw / "6.4. Production_of_Mineral_Raw_Materials_of_individual_Countries_by_Minerals.xlsx"

baci_db = out / "baci.db"
wgi_db = out / "wgi.db"
wmd_db = out / "world_mining_data.db"

print(f"Project folder: {root}")
print(f"Output folder:  {out}")


Project folder: /Users/anishkoyamparambath/Documents/Github/geopolrisk-py/tools
Output folder:  /Users/anishkoyamparambath/Documents/Github/geopolrisk-py/tools/Outputs


In [2]:
# Title: Conversion settings
# Documentation: Keep the small conversion choices in one place.
hs_codes = {
    "250310", "250390", "250410", "250490", "250700", "250810", "250830",
    "250840", "251020", "251110", "251200", "251910", "252010", "252400",
    "252890", "252910", "252922", "260111", "260112", "260200", "260300",
    "260400", "260600", "260700", "260800", "260900", "261000", "261100",
    "261210", "261390", "261400", "261510", "261610", "261690", "261710",
    "270111", "270112", "270119", "270210", "270220", "270900", "271111",
    "271121", "280450", "280480", "280490", "280530", "280540", "282200",
    "282520", "282530", "283030", "283691", "284610", "284690", "710812",
    "711011", "711019", "711021", "711029", "711031", "711039", "711290",
    "711510", "720292", "720293", "760110", "810110", "810310", "810510",
    "810600", "810710", "811211", "811230",
}
wgi_sheet = "pv"
chunk_size = 250_000

country_file = baci_raw / "country_codes_V202601.csv"
product_file = baci_raw / "product_codes_HS92_V202601.csv"

# The app's existing SQL used these table names, so the new raw lookup files are
# written back into the same names for compatibility.
country_table = "country_codes_V202401b"
product_table = "product_codes_HS92_V202401b"


In [3]:
# Title: Small helpers
# Documentation: Quote SQLite names and normalize country names for matching.
def q(name):
    """Quote a SQLite table or column name."""
    return '"' + str(name).replace('"', '""') + '"'


def tidy(name):
    """Return a loose ASCII key for matching country names."""
    text = str(name).strip()
    try:
        text = text.encode("latin1").decode("utf-8")
    except UnicodeError:
        pass
    text = text.replace("’", "'").replace("‘", "'").replace("`", "'")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    text = text.lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def country_code(name):
    """Return the BACI numeric code and ISO3 code for a country name."""
    key = tidy(name)
    if key not in country_lookup:
        raise KeyError(f"No country code found for {name!r}")
    return country_lookup[key]


In [4]:
# Title: Load BACI lookup tables
# Documentation: Read country and product lookup files once for all databases.
country = pd.read_csv(country_file, dtype=str).fillna("")
product = pd.read_csv(product_file, dtype=str).fillna("")

country_lookup = {
    tidy(row.country_name): (row.country_code, row.country_iso3)
    for row in country.itertuples(index=False)
}

# Names below appear in WGI or World Mining but differ from the BACI names.
# "NA" means the old output kept the row but had no matching BACI numeric code.
country_fix = {
    "Bahamas, The": ("44", "BHS"),
    "Bolivia": ("68", "BOL"),
    "Bosnia and Herzegovina": ("70", "BIH"),
    "Brunei": ("96", "BRN"),
    "Cape Verde": ("132", "CPV"),
    "Cayman Islands": ("136", "CYM"),
    "Central African Republic": ("140", "CAF"),
    "Christmas Island": ("162", "CXR"),
    "Congo, Dem. Rep.": ("180", "COD"),
    "Congo, D.R.": ("180", "COD"),
    "Congo, Rep.": ("178", "COG"),
    "Cook Islands": ("184", "COK"),
    "Dominican Republic": ("214", "DOM"),
    "Egypt, Arab Rep.": ("818", "EGY"),
    "Federal Republic of Somalia": ("706", "SOM"),
    "French Guiana": ("328", "GUY"),
    "Gambia, The": ("270", "GMB"),
    "Hong Kong SAR, China": ("344", "HKG"),
    "Iran, Islamic Rep.": ("364", "IRN"),
    "Jersey, Channel Islands": ("NA", "JEY"),
    "Korea, Dem. People's Rep.": ("408", "PRK"),
    "Korea, North": ("408", "PRK"),
    "Korea, Rep.": ("410", "KOR"),
    "Korea, South": ("410", "KOR"),
    "Kosovo": ("697", "R20"),
    "Kyrgyz Republic": ("417", "KGZ"),
    "Lao PDR": ("418", "LAO"),
    "Laos": ("418", "LAO"),
    "Liechtenstein": ("NA", "LIE"),
    "Macao SAR, China": ("446", "MAC"),
    "Marshall Islands": ("584", "MHL"),
    "Martinique": ("NA", "MTQ"),
    "Micronesia, Fed. Sts.": ("583", "FSM"),
    "Moldova": ("498", "MDA"),
    "Monaco": ("NA", "MCO"),
    "Netherlands Antilles (former)": ("530", "ANT"),
    "Puerto Rico (U.S.)": ("NA", "PRI"),
    "Reunion": ("NA", "REU"),
    "Russia": ("643", "RUS"),
    "Slovak Republic": ("703", "SVK"),
    "Solomon Islands": ("90", "SLB"),
    "St. Kitts and Nevis": ("659", "KNA"),
    "St. Lucia": ("662", "LCA"),
    "St. Vincent and the Grenadines": ("670", "VCT"),
    "Syrian Arab Republic": ("760", "SYR"),
    "Taiwan": ("490", "S19"),
    "Taiwan, China": ("NA", "TWN"),
    "Tanzania": ("834", "TZA"),
    "Total": ("DELETE", "DELETE"),
    "Turkiye": ("792", "TUR"),
    "United States": ("842", "USA"),
    "Venezuela, RB": ("862", "VEN"),
    "Vietnam": ("704", "VNM"),
    "Virgin Islands (U.S.)": ("92", "VGB"),
    "West Bank and Gaza": ("275", "PSE"),
    "Yemen, Rep.": ("887", "YEM"),
}
country_lookup.update({tidy(name): value for name, value in country_fix.items()})

print(f"Countries loaded: {len(country):,}")
print(f"Products loaded:  {len(product):,}")


Countries loaded: 238
Products loaded:  5,022


In [5]:
# Title: Update WGI database
# Documentation: Keep existing WGI columns as they are and add only new year columns.
wgi = pd.read_excel(wgi_file, sheet_name=wgi_sheet)
wgi = wgi.rename(
    columns={
        "Economy (name)": "Country/Territory",
        "Economy (code)": "SourceCode",
        "Number of sources": "NumSrc",
        "Governance estimate (approx. -2.5 to +2.5)": "Estimate",
        "Standard error (estimate)": "StdErr",
        "Lower threshold (90% conf. int. score)": "Lower",
        "Upper threshold (90% conf. int. score)": "Upper",
        "Governance score (0-100)": "Rank",
    }
)

wgi["Year"] = wgi["Year"].astype(int).astype(str)
wgi["Country/Territory"] = wgi["Country/Territory"].astype(str).str.strip()
wgi[["country_code", "Code"]] = wgi["Country/Territory"].apply(
    lambda name: pd.Series(country_code(name))
)
wgi["Normalized"] = ((2.5 - pd.to_numeric(wgi["Estimate"], errors="coerce")) / 5).clip(0, 1)

years = sorted(wgi["Year"].unique(), key=int)


def wide(col):
    """Pivot one WGI measure into the output table shape."""
    data = wgi.pivot_table(
        index=["Country/Territory", "Code", "country_code"],
        columns="Year",
        values=col,
        aggfunc="first",
    ).reset_index()
    data.columns = [str(col) for col in data.columns]
    data = data[["Country/Territory", "Code", *years, "country_code"]]
    return data.astype(str)


def row_key(row, code_col="Code", country_col="Country/Territory"):
    """Use the stable country code first, then ISO/code, then country name."""
    cc = str(row.get("country_code", "")).strip()
    code = str(row.get(code_col, "")).strip()
    name = str(row.get(country_col, "")).strip()
    if cc and cc.lower() not in {"na", "nan", "none"}:
        return f"cc:{cc}"
    if code and code.lower() not in {"na", "nan", "none"}:
        return f"code:{code}"
    return f"name:{tidy(name)}"


def update_table(con, table, fresh):
    """Add new WGI years without changing existing year columns."""
    exists = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table,),
    ).fetchone()
    if not exists:
        fresh.to_sql(table, con, if_exists="replace", index=False)
        print(f"{table}: created with {len(fresh):,} rows")
        return

    old = pd.read_sql_query(f"SELECT * FROM {q(table)}", con).astype(str)
    old_cols = list(old.columns)
    fresh_cols = list(fresh.columns)
    fresh_years = [col for col in fresh_cols if re.fullmatch(r"\d{4}", str(col))]
    new_years = [year for year in fresh_years if year not in old_cols]

    insert_at = old_cols.index("country_code") if "country_code" in old_cols else len(old_cols)
    final_cols = old_cols[:insert_at] + new_years + old_cols[insert_at:]

    old["_key"] = old.apply(row_key, axis=1)
    fresh = fresh.copy()
    fresh["_key"] = fresh.apply(row_key, axis=1)
    fresh_one = fresh.drop_duplicates("_key").set_index("_key")

    for year in new_years:
        old[year] = old["_key"].map(fresh_one[year])

    missing = fresh[~fresh["_key"].isin(old["_key"])]
    if not missing.empty:
        rows = []
        for _, row in missing.drop_duplicates("_key").iterrows():
            item = {col: "" for col in final_cols}
            for col in final_cols:
                if col in row:
                    item[col] = row[col]
            rows.append(item)
        for item in rows:
            old.loc[len(old), final_cols] = [item.get(col, "") for col in final_cols]

    result = old.drop(columns=["_key"])[final_cols]
    result.to_sql(table, con, if_exists="replace", index=False)
    print(f"{table}: kept {len(old_cols)} columns, added {new_years or 'no new years'}")


wgi_tables = {
    "Estimate": "Estimate",
    "StdErr": "StdErr",
    "NumSrc": "NumSrc",
    "Rank": "Rank",
    "Lower": "Lower",
    "Upper": "Upper",
    "Normalized": "Normalized",
}

with sqlite3.connect(wgi_db) as con:
    for table, col in wgi_tables.items():
        update_table(con, table, wide(col))


Estimate: kept 29 columns, added no new years
StdErr: kept 29 columns, added no new years
NumSrc: kept 29 columns, added no new years
Rank: kept 29 columns, added no new years
Lower: kept 29 columns, added no new years
Upper: kept 29 columns, added no new years
Normalized: kept 29 columns, added no new years


In [ ]:
# Title: Create BACI trade database
# Documentation: Fully replace BACI trade rows, keeping only the HS codes used by the original app database.
files = sorted(baci_raw.glob("BACI_HS92_Y*_V*.csv"))
if not files:
    raise FileNotFoundError(f"No BACI yearly files found in {baci_raw}")

with sqlite3.connect(baci_db) as con:
    for view in ["v_baci_trade_with_wgi", "v_baci_year_count", "v_wgi_year_country"]:
        con.execute(f"DROP VIEW IF EXISTS {q(view)}")
    con.execute("DROP TABLE IF EXISTS baci_trade")
    con.execute(
        """
        CREATE TABLE baci_trade (
            t TEXT,
            i TEXT,
            j TEXT,
            k INTEGER,
            v TEXT,
            q TEXT
        )
        """
    )

    country.to_sql(country_table, con, if_exists="replace", index=False)
    product.to_sql(product_table, con, if_exists="replace", index=False)

    for file in files:
        kept = 0
        for chunk in pd.read_csv(file, dtype=str, chunksize=chunk_size):
            chunk = chunk[["t", "i", "j", "k", "v", "q"]]
            key = chunk["k"].str.zfill(6)
            part = chunk[key.isin(hs_codes)].copy()
            if not part.empty:
                part["k"] = part["k"].astype(int)
                part.to_sql("baci_trade", con, if_exists="append", index=False)
                kept += len(part)
        print(f"{file.name}: {kept:,} rows kept")


BACI_HS92_Y1995_V202601.csv: 26,496 rows kept
BACI_HS92_Y1996_V202601.csv: 27,959 rows kept
BACI_HS92_Y1997_V202601.csv: 28,939 rows kept
BACI_HS92_Y1998_V202601.csv: 29,530 rows kept
BACI_HS92_Y1999_V202601.csv: 29,613 rows kept
BACI_HS92_Y2000_V202601.csv: 34,761 rows kept
BACI_HS92_Y2001_V202601.csv: 35,137 rows kept
BACI_HS92_Y2002_V202601.csv: 35,880 rows kept
BACI_HS92_Y2003_V202601.csv: 36,392 rows kept
BACI_HS92_Y2004_V202601.csv: 38,125 rows kept
BACI_HS92_Y2005_V202601.csv: 38,762 rows kept
BACI_HS92_Y2006_V202601.csv: 39,891 rows kept
BACI_HS92_Y2007_V202601.csv: 41,736 rows kept
BACI_HS92_Y2008_V202601.csv: 42,868 rows kept
BACI_HS92_Y2009_V202601.csv: 42,140 rows kept
BACI_HS92_Y2010_V202601.csv: 44,568 rows kept
BACI_HS92_Y2011_V202601.csv: 46,288 rows kept
BACI_HS92_Y2012_V202601.csv: 47,226 rows kept
BACI_HS92_Y2013_V202601.csv: 47,126 rows kept


In [ ]:
# Title: Add BACI views
# Documentation: Copy WGI Normalized into BACI and create views with the original app-compatible names.
with sqlite3.connect(wgi_db) as src:
    norm = pd.read_sql_query("SELECT * FROM Normalized", src)

year_cols = [col for col in norm.columns if re.fullmatch(r"\d{4}", str(col))]

with sqlite3.connect(baci_db) as con:
    for view in ["v_baci_trade_with_wgi", "v_baci_year_count", "v_wgi_year_country"]:
        con.execute(f"DROP VIEW IF EXISTS {q(view)}")

    norm.to_sql("Normalized", con, if_exists="replace", index=False)

    parts = []
    for i, year in enumerate(year_cols):
        start = "SELECT" if i == 0 else "UNION ALL SELECT"
        parts.append(
            f"{start} '{year}' AS Year, {q('Country/Territory')}, Code, {q(year)} AS wgi, country_code FROM Normalized"
        )
    con.execute(
        """
        CREATE VIEW v_wgi_year_country AS
        SELECT
            Year,
            "Country/Territory",
            Code,
            country_code,
            wgi
        FROM (
        """
        + "\n        ".join(parts)
        + """
        ) AS UnpivotedData
        ORDER BY "Country/Territory", Year
        """
    )

    con.execute(
        """
        CREATE VIEW v_baci_year_count AS
        SELECT baci.t as year, count(*)
        FROM baci_trade baci
        GROUP BY YEAR
        """
    )

    con.execute(
        """
        CREATE VIEW v_baci_trade_with_wgi AS
        select
            bacitab.t as period,
            bacitab.j as reporterCode,
            (SELECT cc.country_name FROM country_codes_V202401b cc WHERE bacitab.j = cc.country_code) AS reporterDesc,
            (SELECT cc.country_iso3 FROM country_codes_V202401b cc WHERE bacitab.j = cc.country_code) AS reporterISO,
            bacitab.i as partnerCode,
            (SELECT cc.country_name FROM country_codes_V202401b cc WHERE bacitab.i = cc.country_code) AS partnerDesc,
            (SELECT cc.country_iso3 FROM country_codes_V202401b cc WHERE bacitab.i = cc.country_code) AS partnerISO,
            bacitab.k as cmdCode,
            TRIM(bacitab.q) as qty,
            TRIM(bacitab.v) as cifvalue,
            (SELECT vwyc.wgi FROM v_wgi_year_country vwyc WHERE bacitab.t = vwyc.Year and bacitab.i = vwyc.country_code) AS partnerWGI
        from baci_trade bacitab
        """
    )

print(f"WGI years in BACI view: {', '.join(year_cols)}")


WGI years in BACI view: 1996, 1998, 2000, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024


In [ ]:
# Title: Update World Mining tables
# Documentation: Keep existing mineral columns as they are and add only new year columns.
wmd_book = pd.ExcelFile(wmd_file)
wmd_tables = []
wmd_logs = []


def mining_key(row):
    """Use the stable country code first, then ISO code, then country name."""
    cc = str(row.get("Country_Code", "")).strip()
    iso = str(row.get("Country_ISO", "")).strip()
    name = str(row.get("Country", "")).strip()
    if cc and cc.lower() not in {"na", "nan", "none"}:
        return f"cc:{cc}"
    if iso and iso.lower() not in {"na", "nan", "none"}:
        return f"iso:{iso}"
    return f"name:{tidy(name)}"


def read_mining(sheet):
    """Read one mineral sheet and add BACI country fields."""
    df = pd.read_excel(wmd_file, sheet_name=sheet, header=1)
    df = df.dropna(how="all")
    df.columns = [str(col).strip() for col in df.columns]
    df["Country"] = df["Country"].astype(str).str.strip()
    df = df[(df["Country"] != "") & (df["Country"].str.lower() != "nan")]

    years = [col for col in df.columns if re.fullmatch(r"\d{4}", col)]
    df = df[["Country", "unit", *years, "data source"]]

    codes = df["Country"].apply(country_code)
    df.insert(1, "Country_Code", codes.apply(lambda item: item[0]))
    df.insert(2, "Country_ISO", codes.apply(lambda item: item[1]))
    df = df.rename(columns={"data source": "data_source"})
    return df[["Country", "Country_Code", "Country_ISO", *years, "unit", "data_source"]]


def update_mining(con, table, fresh):
    """Add new mining years without changing existing year columns."""
    exists = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table,),
    ).fetchone()
    if not exists:
        fresh.to_sql(table, con, if_exists="replace", index=False)
        print(f"{table}: created with {len(fresh):,} rows")
        return

    old = pd.read_sql_query(f"SELECT * FROM {q(table)}", con)
    old_cols = list(old.columns)
    fresh_cols = list(fresh.columns)
    fresh_years = [col for col in fresh_cols if re.fullmatch(r"\d{4}", str(col))]
    new_years = [year for year in fresh_years if year not in old_cols]

    insert_at = old_cols.index("unit") if "unit" in old_cols else len(old_cols)
    final_cols = old_cols[:insert_at] + new_years + old_cols[insert_at:]

    old["_key"] = old.apply(mining_key, axis=1)
    fresh = fresh.copy()
    fresh["_key"] = fresh.apply(mining_key, axis=1)
    fresh_one = fresh.drop_duplicates("_key").set_index("_key")

    for year in new_years:
        old[year] = old["_key"].map(fresh_one[year])

    missing = fresh[~fresh["_key"].isin(old["_key"])]
    if not missing.empty:
        rows = []
        for _, row in missing.drop_duplicates("_key").iterrows():
            item = {col: None for col in final_cols}
            for col in final_cols:
                if col in row:
                    item[col] = row[col]
            rows.append(item)
        for item in rows:
            old.loc[len(old), final_cols] = [item.get(col, "") for col in final_cols]

    result = old.drop(columns=["_key"])[final_cols]
    result.to_sql(table, con, if_exists="replace", index=False)
    print(f"{table}: kept {len(old_cols)} columns, added {new_years or 'no new years'}")


with sqlite3.connect(wmd_db) as con:
    for view in ["v_sum_by_hs_and_jear", "v_all_world_mining_data"]:
        con.execute(f"DROP VIEW IF EXISTS {q(view)}")

    iso = country[["country_code", "country_name"]].copy()
    iso.columns = ["ISO", "Country"]
    iso.to_sql("Country_ISO", con, if_exists="replace", index=False)

    for sheet in wmd_book.sheet_names:
        table = sheet.strip()
        fresh = read_mining(sheet)
        update_mining(con, table, fresh)
        wmd_tables.append(table)
        wmd_logs.append(
            {
                "sheet_name": table,
                "log_message": f"Updated from {wmd_file.name}",
                "log_update": 1,
            }
        )

    pd.DataFrame(wmd_logs).to_sql("logging", con, if_exists="replace", index=False)


Iron (Fe): kept 12 columns, added no new years
Chromium (Cr2O3): kept 12 columns, added no new years
Cobalt: kept 12 columns, added no new years
Manganese: kept 12 columns, added no new years
Molybdenum: kept 12 columns, added no new years
Nickel: kept 12 columns, added no new years
Niobium (Nb2O5): kept 12 columns, added no new years
Tantalum (Ta2O5): kept 12 columns, added no new years
Titanium (TiO2): kept 12 columns, added no new years
Tungsten (W): kept 12 columns, added no new years
Vanadium (V): kept 12 columns, added no new years
Aluminium: kept 12 columns, added no new years
Antimony: kept 12 columns, added no new years
Arsenic: kept 12 columns, added no new years
Bauxite: kept 12 columns, added no new years
Beryllium (conc.): kept 12 columns, added no new years
Bismuth: kept 12 columns, added no new years
Cadmium: kept 12 columns, added no new years
Copper: kept 12 columns, added no new years
Gallium: kept 12 columns, added no new years
Germanium: kept 12 columns, added no ne

In [ ]:
# Title: Create World Mining views
# Documentation: Build the union view and yearly summary view from the updated table columns.
if "wmd_tables" not in globals() or not wmd_tables:
    wmd_tables = [sheet.strip() for sheet in pd.ExcelFile(wmd_file).sheet_names]

with sqlite3.connect(wmd_db) as con:
    for view in ["v_sum_by_hs_and_jear", "v_all_world_mining_data"]:
        con.execute(f"DROP VIEW IF EXISTS {q(view)}")

    union_sql = " UNION ALL\n".join(
        f"SELECT '{table}' AS HS, * FROM {q(table)}" for table in wmd_tables
    )
    con.execute("CREATE VIEW v_all_world_mining_data AS\n" + union_sql)

    first_cols = [row[1] for row in con.execute(f"PRAGMA table_info({q(wmd_tables[0])})")]
    year_cols = [col for col in first_cols if re.fullmatch(r"\d{4}", col)]
    sum_cols = ",\n            ".join(
        f"SUM({q(year)}) AS {q(year + ' / metr. t')}" for year in year_cols
    )
    con.execute(
        f"""
        CREATE VIEW v_sum_by_hs_and_jear AS
        SELECT
            HS,
            {sum_cols}
        FROM v_all_world_mining_data
        GROUP BY HS
        """
    )

print(f"World Mining views use years: {', '.join(year_cols)}")


World Mining views use years: 2018, 2019, 2020, 2021, 2022, 2023, 2024


In [ ]:
# Title: Check database outputs
# Documentation: Print table counts and a few row counts to confirm the rebuild worked.
for db in [wgi_db, baci_db, wmd_db]:
    print(f"\n{db.name}")
    with sqlite3.connect(db) as con:
        tables = [
            row[0]
            for row in con.execute(
                "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
            )
        ]
        views = [
            row[0]
            for row in con.execute(
                "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
            )
        ]
        print(f"tables: {len(tables)}")
        print(f"views:  {len(views)}")
        for table in tables[:8]:
            count = con.execute(f"SELECT COUNT(*) FROM {q(table)}").fetchone()[0]
            print(f"  {table}: {count:,} rows")



wgi.db
tables: 7
views:  0
  Estimate: 217 rows
  Lower: 217 rows
  Normalized: 217 rows
  NumSrc: 217 rows
  Rank: 217 rows
  StdErr: 217 rows
  Upper: 217 rows

baci.db
tables: 4
views:  3
  Normalized: 217 rows
  baci_trade: 20,472,033 rows
  country_codes_V202401b: 238 rows
  product_codes_HS92_V202401b: 5,022 rows

world_mining_data.db
tables: 68
views:  2
  Aluminium: 42 rows
  Antimony: 19 rows
  Arsenic: 8 rows
  Asbestos: 5 rows
  Baryte: 33 rows
  Bauxite: 35 rows
  Bentonite: 53 rows
  Beryllium (conc.): 10 rows
